In [ ]:
!bash scripts/prereq.sh

In [ ]:
%pip install -U -r requirements.txt -q

In [6]:
# Import libraries
import boto3
from boto3.session import Session

from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS

from strands.tools import tool

In [5]:
# Get boto session
boto_session = Session()
region = boto_session.region_name
#region

'us-west-2'

In [7]:
# Get return policy
@tool
def get_return_policy(product_category: str) -> str:
    """
    Get return policy information for a specific product category.

    Args:
        product_category: Electronics category (e.g., 'smartphones', 'laptops', 'accessories')

    Returns:
        Formatted return policy details including timeframes and conditions
    """
    # Mock return policy database - in real implementation, this would query policy database
    return_policies = {
        "smartphones": {
            "window": "30 days",
            "condition": "Original packaging, no physical damage, factory reset required",
            "process": "Online RMA portal or technical support",
            "refund_time": "5-7 business days after inspection",
            "shipping": "Free return shipping, prepaid label provided",
            "warranty": "1-year manufacturer warranty included",
        },
        "laptops": {
            "window": "30 days",
            "condition": "Original packaging, all accessories, no software modifications",
            "process": "Technical support verification required before return",
            "refund_time": "7-10 business days after inspection",
            "shipping": "Free return shipping with original packaging",
            "warranty": "1-year manufacturer warranty, extended options available",
        },
        "accessories": {
            "window": "30 days",
            "condition": "Unopened packaging preferred, all components included",
            "process": "Online return portal",
            "refund_time": "3-5 business days after receipt",
            "shipping": "Customer pays return shipping under $50",
            "warranty": "90-day manufacturer warranty",
        },
    }

    # Default policy for unlisted categories
    default_policy = {
        "window": "30 days",
        "condition": "Original condition with all included components",
        "process": "Contact technical support",
        "refund_time": "5-7 business days after inspection",
        "shipping": "Return shipping policies vary",
        "warranty": "Standard manufacturer warranty applies",
    }

    policy = return_policies.get(product_category.lower(), default_policy)
    return (
        f"Return Policy - {product_category.title()}:\n\n"
        f"• Return window: {policy['window']} from delivery\n"
        f"• Condition: {policy['condition']}\n"
        f"• Process: {policy['process']}\n"
        f"• Refund timeline: {policy['refund_time']}\n"
        f"• Shipping: {policy['shipping']}\n"
        f"• Warranty: {policy['warranty']}"
    )


print("✅ Return policy tool ready")

✅ Return policy tool ready


In [8]:
# Get product information
@tool
def get_product_info(product_type: str) -> str:
    """
    Get detailed technical specifications and information for electronics products.

    Args:
        product_type: Electronics product type (e.g., 'laptops', 'smartphones', 'headphones', 'monitors')
    Returns:
        Formatted product information including warranty, features, and policies
    """
    # Mock product catalog - in real implementation, this would query a product database
    products = {
        "laptops": {
            "warranty": "1-year manufacturer warranty + optional extended coverage",
            "specs": "Intel/AMD processors, 8-32GB RAM, SSD storage, various display sizes",
            "features": "Backlit keyboards, USB-C/Thunderbolt, Wi-Fi 6, Bluetooth 5.0",
            "compatibility": "Windows 11, macOS, Linux support varies by model",
            "support": "Technical support and driver updates included",
        },
        "smartphones": {
            "warranty": "1-year manufacturer warranty",
            "specs": "5G/4G connectivity, 128GB-1TB storage, multiple camera systems",
            "features": "Wireless charging, water resistance, biometric security",
            "compatibility": "iOS/Android, carrier unlocked options available",
            "support": "Software updates and technical support included",
        },
        "headphones": {
            "warranty": "1-year manufacturer warranty",
            "specs": "Wired/wireless options, noise cancellation, 20Hz-20kHz frequency",
            "features": "Active noise cancellation, touch controls, voice assistant",
            "compatibility": "Bluetooth 5.0+, 3.5mm jack, USB-C charging",
            "support": "Firmware updates via companion app",
        },
        "monitors": {
            "warranty": "3-year manufacturer warranty",
            "specs": "4K/1440p/1080p resolutions, IPS/OLED panels, various sizes",
            "features": "HDR support, high refresh rates, adjustable stands",
            "compatibility": "HDMI, DisplayPort, USB-C inputs",
            "support": "Color calibration and technical support",
        },
    }
    product = products.get(product_type.lower())
    if not product:
        return f"Technical specifications for {product_type} not available. Please contact our technical support team for detailed product information and compatibility requirements."

    return (
        f"Technical Information - {product_type.title()}:\n\n"
        f"• Warranty: {product['warranty']}\n"
        f"• Specifications: {product['specs']}\n"
        f"• Key Features: {product['features']}\n"
        f"• Compatibility: {product['compatibility']}\n"
        f"• Support: {product['support']}"
    )


print("✅ get_product_info tool ready")

✅ get_product_info tool ready


In [9]:
#  This tool allows customers to get troubleshooting support or suggestions on product recommendations etc.

@tool
def web_search(keywords: str, region: str = "us-en", max_results: int = 5) -> str:
    """Search the web for updated information.

    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.

    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "Rate limit reached. Please try again later."
    except DDGSException as e:
        return f"Search error: {e}"
    except Exception as e:
        return f"Search error: {str(e)}"


print("✅ Web search tool ready")

✅ Web search tool ready


In [12]:
# Customer Support Agent - Knowledge Base Integration Steps
import os

def download_files():
    # Get account and region
    account_id = boto3.client("sts").get_caller_identity()["Account"]
    region = boto3.Session().region_name
    bucket_name = f"{account_id}-{region}-kb-data-bucket"

    # Create local folder
    os.makedirs("knowledge_base_data", exist_ok=True)

    # Download all files
    s3 = boto3.client("s3")
    objects = s3.list_objects_v2(Bucket=bucket_name)

    for obj in objects["Contents"]:
        file_name = obj["Key"]
        s3.download_file(bucket_name, file_name, f"knowledge_base_data/{file_name}")
        print(f"Downloaded: {file_name}")

    print("All files saved to: knowledge_base_data/")


# Run it
download_files()


Downloaded: laptop-maintenance-guide.txt
Downloaded: monitor-calibration-guide.txt
Downloaded: smartphone-setup-guide.txt
Downloaded: troubleshooting-guide.txt
Downloaded: warranty-service-guide.txt
Downloaded: wireless-connectivity-guide.txt
All files saved to: knowledge_base_data/


In [13]:
# Sync the knowledge base with product technical_support files from S3 which can be integrated with the agent

import time

# Get parameters
ssm = boto3.client("ssm")
bedrock = boto3.client("bedrock-agent")
s3 = boto3.client("s3")

account_id = boto3.client("sts").get_caller_identity()["Account"]
region = boto3.Session().region_name

kb_id = ssm.get_parameter(Name=f"/{account_id}-{region}/kb/knowledge-base-id")[
    "Parameter"
]["Value"]
ds_id = ssm.get_parameter(Name=f"/{account_id}-{region}/kb/data-source-id")[
    "Parameter"
]["Value"]

# Get file names from S3 bucket
bucket_name = f"{account_id}-{region}-kb-data-bucket"
s3_objects = s3.list_objects_v2(Bucket=bucket_name)
file_names = [obj["Key"] for obj in s3_objects.get("Contents", [])]

# Start sync job
response = bedrock.start_ingestion_job(
    knowledgeBaseId=kb_id, dataSourceId=ds_id, description="Quick sync"
)

job_id = response["ingestionJob"]["ingestionJobId"]
print("Bedrock knowledge base sync job started, ingesting the data files from s3")

# Monitor until complete
while True:
    job = bedrock.get_ingestion_job(
        knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=job_id
    )["ingestionJob"]

    status = job["status"]

    if status in ["COMPLETE", "FAILED"]:
        break

    time.sleep(10)

# Print final result
if status == "COMPLETE":
    file_count = job.get("statistics", {}).get("numberOfDocumentsScanned", 0)
    files_list = ", ".join(file_names)
    print(
        f"Bedrock knowledge base sync job completed Successfully, ingested {file_count} files"
    )
    print(f"Files ingested: {files_list}")
else:
    print(f"Bedrock knowledge base sync job failed with status: {status}")

Bedrock knowledge base sync job started, ingesting the data files from s3
Bedrock knowledge base sync job completed Successfully, ingested 6 files
Files ingested: laptop-maintenance-guide.txt, monitor-calibration-guide.txt, smartphone-setup-guide.txt, troubleshooting-guide.txt, warranty-service-guide.txt, wireless-connectivity-guide.txt


In [14]:
# This tool provides customers with comprehensive technical support and troubleshooting assistance by accessing our knowledge base of 
# electronics documentation. It includes detailed setup guides, maintenance instructions, troubleshooting steps, connectivity solutions, 
# and warranty service information. This tool helps customers resolve technical issues, properly configure their devices, and understand 
# maintenance requirements for optimal product performance.

from strands.models import BedrockModel
from strands import Agent
from strands_tools import retrieve


@tool
def get_technical_support(issue_description: str) -> str:
    try:
        # Get KB ID from parameter store
        ssm = boto3.client("ssm")
        account_id = boto3.client("sts").get_caller_identity()["Account"]
        region = boto3.Session().region_name

        kb_id = ssm.get_parameter(Name=f"/{account_id}-{region}/kb/knowledge-base-id")[
            "Parameter"
        ]["Value"]
        print(f"Successfully retrieved KB ID: {kb_id}")

        # Use strands retrieve tool
        tool_use = {
            "toolUseId": "tech_support_query",
            "input": {
                "text": issue_description,
                "knowledgeBaseId": kb_id,
                "region": region,
                "numberOfResults": 3,
                "score": 0.4,
            },
        }

        result = retrieve.retrieve(tool_use)

        if result["status"] == "success":
            return result["content"][0]["text"]
        else:
            return f"Unable to access technical support documentation. Error: {result['content'][0]['text']}"

    except Exception as e:
        print(f"Detailed error in get_technical_support: {str(e)}")
        return f"Unable to access technical support documentation. Error: {str(e)}"


print("✅ Technical support tool ready")

✅ Technical support tool ready


In [15]:
# Create and Configure the Customer Support Agent
SYSTEM_PROMPT = """You are a helpful and professional customer support assistant for an electronics e-commerce company.
Your role is to:
- Provide accurate information using the tools available to you
- Support the customer with technical information and product specifications, and maintenance questions
- Be friendly, patient, and understanding with customers
- Always offer additional help after answering questions
- If you can't help with something, direct customers to the appropriate contact

You have access to the following tools:
1. get_return_policy() - For warranty and return policy questions
2. get_product_info() - To get information about a specific product
3. web_search() - To access current technical documentation, or for updated information. 
4. get_technical_support() - For troubleshooting issues, setup guides, maintenance tips, and detailed technical assistance
For any technical problems, setup questions, or maintenance concerns, always use the get_technical_support() tool as it contains our comprehensive technical documentation and step-by-step guides.

Always use the appropriate tool to get accurate, up-to-date information rather than making assumptions about electronic products or specifications."""

# Initialize the Bedrock model (Amazon Nova 2 Lite)
model = BedrockModel(
    model_id="us.amazon.nova-2-lite-v1:0",
    temperature=0.3,
    region_name=region,
)

# Create the customer support agent with all tools
agent = Agent(
    model=model,
    tools=[
        get_product_info,  # Tool 1: Simple product information lookup
        get_return_policy,  # Tool 2: Simple return policy lookup
        web_search,  # Tool 3: Access the web for updated information
        get_technical_support,  # Tool 4: Technical support & troubleshooting
    ],
    system_prompt=SYSTEM_PROMPT,
)

print("Customer Support Agent created successfully!")

Customer Support Agent created successfully!


In [16]:
# Let's test our agent with sample queries to ensure all tools work correctly.
response = agent("What's the return policy for my thinkpad X1 Carbon?")


Tool #1: get_return_policy
The return policy for your ThinkPad X1 Carbon (laptop category) is as follows:

**Return Policy for Laptops:**
- **Return Window:** 30 days from delivery
- **Condition:** Must be in original packaging with all accessories, and no software modifications
- **Process:** Technical support verification is required before processing your return
- **Refund Timeline:** 7-10 business days after inspection
- **Shipping:** Free return shipping when using original packaging
- **Warranty:** Comes with a 1-year manufacturer warranty, with extended warranty options available

If you need to initiate a return, I recommend contacting our technical support team first for verification. Would you like me to help you connect with technical support or do you have any other questions about your ThinkPad?

In [17]:
response = agent("My laptop won't turn on, what should I check?")


/tmp/ipykernel_1663/3914750932.py:36: DeprecationWarning: retrieve is deprecated. This warning becomes an error log in v0.9.0. Migration path: use MemoryManager with BedrockKnowledgeBaseStore(writable=False). Note the store applies no relevance-score floor, so filter results yourself. See https://strandsagents.com/docs/user-guide/concepts/memory/overview/
  result = retrieve.retrieve(tool_use)
DEPRECATION WARNING: retrieve is deprecated. This warning becomes an error log in v0.9.0. Migration path: use MemoryManager with BedrockKnowledgeBaseStore(writable=False). Note the store applies no relevance-score floor, so filter results yourself. See https://strandsagents.com/docs/user-guide/concepts/memory/overview/



Tool #2: get_technical_support
Successfully retrieved KB ID: AA3OGUUIZB
Based on the troubleshooting guide, here are the key steps to check when your laptop won't turn on:

## Power Issues Troubleshooting Steps:

1. **Check Power Cable Connections**
   - Ensure the power cable is properly connected to both your laptop and a working power outlet
   - Try plugging the cable into a different outlet to verify the outlet is functioning

2. **Verify Power Outlet Functionality**
   - Test the outlet with another device to confirm it's providing power

3. **Try Different Power Cable (if available)**
   - If you have access to another power cable, try using that to rule out a faulty cable

4. **Check for Physical Damage**
   - Inspect the laptop for any visible damage to the power port or cable
   - Look for any loose connections

5. **Check Battery Connection (if applicable)**
   - If your laptop has a removable battery, try removing and reinserting it properly
   - Some laptops function with

In [18]:
response = agent(
    "I bought an iphone 14 last month. I don't like it because it heats up. How do I solve it?"
)

/tmp/ipykernel_1663/3914750932.py:36: DeprecationWarning: retrieve is deprecated. This warning becomes an error log in v0.9.0. Migration path: use MemoryManager with BedrockKnowledgeBaseStore(writable=False). Note the store applies no relevance-score floor, so filter results yourself. See https://strandsagents.com/docs/user-guide/concepts/memory/overview/
  result = retrieve.retrieve(tool_use)
DEPRECATION WARNING: retrieve is deprecated. This warning becomes an error log in v0.9.0. Migration path: use MemoryManager with BedrockKnowledgeBaseStore(writable=False). Note the store applies no relevance-score floor, so filter results yourself. See https://strandsagents.com/docs/user-guide/concepts/memory/overview/



Tool #3: get_technical_support
Successfully retrieved KB ID: AA3OGUUIZB

Tool #4: web_search
Here are several effective solutions to address your iPhone 14 overheating issues:

## **Common Causes of iPhone Overheating**
- High ambient temperatures or direct sunlight exposure
- Resource-intensive apps (gaming, video streaming, GPS)
- Running multiple apps simultaneously
- Charging while using the phone
- Poor ventilation (e.g., placing phone under pillows)

## **Step-by-Step Solutions**

### **Immediate Fixes**
1. **Move to a Cooler Environment**
   - Move away from direct sunlight or heat sources
   - Avoid leaving your iPhone in hot cars or direct sunlight

2. **Stop Using Resource-Heavy Apps**
   - Close apps like games, video editors, GPS navigation, and video streaming
   - Force quit apps: Swipe up from the bottom of the screen and swipe away running apps

3. **Lower Screen Brightness**
   - Reduce brightness to minimum comfortable level
   - Enable Auto-Brightness if available



In [19]:
response = agent(
    "I want to travel Canada"
)

Traveling Canada sounds like a fantastic adventure! Canada is a vast and diverse country with many amazing experiences to offer. To help you plan your trip effectively, I'd like to ask a few questions to understand your interests better:

**To help you plan your Canadian adventure, could you share:**
1. What part of Canada are you most interested in? (e.g., Toronto, Vancouver, Banff, Montreal, Atlantic Canada, etc.)
2. What time of year are you planning to travel? (Winter, spring, summer, or fall each offer different experiences)
3. What are your main interests? (Nature/wildlife, cities/urban experiences, hiking, food, festivals, shopping, etc.)
4. How long are you planning to stay?
5. Do you have any specific activities you're hoping to do? (Hiking, skiing, whale watching, city tours, etc.)

Canada offers incredible experiences whether you're looking for:
- **Nature lovers:** National parks like Banff, Jasper, or the Rocky Mountains
- **City experiences:** Toronto, Vancouver, Montreal